<a href="https://colab.research.google.com/github/kharismansh/Monthly-Report-CS-using-Redash/blob/main/Monthly_Reporting_CS_Redash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Corp Sales Reporting: Monthly**
For monthly report using Redash

In [ ]:
# start_date and end_date
# edit start date and end date here
start_date = '2025-05-01'
end_date = '2025-05-31'

# edit month report here
month = 'May 2025'


In [ ]:
import os
import requests
import time
from pprint import pprint
import json
import pandas as pd
import sys
from datetime import datetime,timedelta
start_time = datetime.now()
import numpy as np

from email.mime.text import MIMEText
from email.mime.application import MIMEApplication
from email.mime.multipart import MIMEMultipart
from smtplib import SMTP
import ssl
import smtplib
!pip install pretty_html_table
from pretty_html_table import build_table

def poll_job(s, redash_url, job):
    # TODO: add timeout
    while job['status'] not in (3,4):
        response = s.get('{}/api/jobs/{}'.format(redash_url, job['id']))
        job = response.json()['job']
        time.sleep(5)

    if job['status'] == 3:
        return job['query_result_id']

    return None


def get_fresh_query_result(redash_url, query_id, api_key, params):
    s = requests.Session()
    s.headers.update({'Authorization': 'Key {}'.format(api_key)})

    payload = dict(max_age=0, parameters=params)

    response = s.post('{}/api/queries/{}/results'.format(redash_url, query_id), data=json.dumps(payload))

    if response.status_code != 200:
        return 'Refresh failed'
        raise Exception('Refresh failed.')

    result_id = poll_job(s, redash_url, response.json()['job'])

    if result_id:
        while True:
            try:
                response = s.get('{}/api/queries/{}/results/{}.json'.format(redash_url, query_id, result_id))
                break
            except:
                print('retry')

        if response.status_code != 200:
            raise Exception('Failed getting results.')
    else:
        raise Exception('Query execution failed.')

    return response.json()['query_result']['data']['rows']

## List Shipper
update list shipper here

In [ ]:
# FORMAT:

list_shipper = [
[132,'Apotek Sarana'],
[454,'APOTEK ANUGERAH'],
[208,'PT Tasti'],
[729,'CV Tadika'],
]

listglobal_shipper_id = []
for i in list_shipper:
  listglobal_shipper_id.append(i[0])

listShipper = ",".join(map(str, listglobal_shipper_id))

# count number of shipper in list
def count_lists(data):
    if isinstance(data, list):
        count = 1
    else:
        count = 0

    if isinstance(data, list) or isinstance(data, dict):
        for value in data.values() if isinstance(data, dict) else data:
            count += count_lists(value)

    return count

list_count = count_lists(list_shipper)
print(listShipper)
# print('Jumlah Shipper :', list_count)

## Pull Data from Redash 245

In [ ]:
print("Number of Shipper in lists:", list_count)
print('>> Pulling data from Redash...')
loopcounter = 0
while True:
    try:
        params = {'creation_end_date':end_date, 'creation_start_date':start_date, 'global_shipper_id':listShipper}
        api_key = 'xxxxx'
        result = get_fresh_query_result('https://redash-id.ninjavan.co/',245, api_key, params)
        print('>> Pulling data SUCCESS')
        break
    except:
        print('Pulling data failed. Retrying..')
        loopcounter = loopcounter +1
        if loopcounter >=5:
            break

raw_data = pd.DataFrame(result)
print("Total Order : ",raw_data.shape[0])
print("Number of Shipper with Order : ",raw_data['global_shipper_id'].nunique())

#### List No Order

In [ ]:
print('Shipper yang tidak ada order')

for i in list(list_shipper):
  data_shipper = raw_data[raw_data['global_shipper_id'] == i[0]]
  if data_shipper.shape[0] != 0 :
    pass
  else:
    # print(i[1], "-", "Total Order : ", data_shipper.shape[0])
    print(i[1])
print('')
print('')
print('Granular Status')
raw_data['granular_status'].unique()

## Adjust Columns

In [ ]:
# adjust columns
raw_data = raw_data[['order_creation_datetime','tracking_id','global_shipper_id','shipper_name','from_address','to_address','rts_flag','granular_status','to_name',
                    'to_contact','instruction','order_comments','first_inbound_datetime','pod_datetime','pod_recipient_name','recipient_relationship','pod_photo_url',
                    'pod_signature_url','first_attempt_at','first_fail_reason','last_attempt_at','last_fail_reason','no_of_attempt','cod_value']]


## Split by Shipper and Send Email

### Email Function
Atur disini buat setting email


*   Edit **`msg['From']`** as From Email
*   Edit **`msg['To']`** as To Email
*   Edit **`to`** for recipients email




In [ ]:
def send_email():
        msg = MIMEMultipart()
        msg['Subject'] = 'Monthly Report '+i[1]+' Periode ' +month
        msg['From'] = 'Kharisma Nur’aisyah'

# Insert LIST Recipient
        msg['To'] = 'xxx <xxx@gmail.com>'
        msg['Cc'] = 'xxx <xxx@gmail.com>'

        filename = filenames
        fp = open(filename,'rb')
        attachment = MIMEApplication(fp.read(),_subtype="xlsx")
        fp.close()
        attachment.add_header('Content-Disposition', 'attachment', filename=filename)
        msg.attach(attachment)

        html = """\
        <html>
            <body>
            <p>Dear Corporate Sales Team,</p>

        <p>Berikut kami lampirkan Monthly Report {0} periode {2}</p>
        {1}
        <p><b> Regards,
        <br>Kharisma Nur'aisyah</b></p>
            </body>
        </html>
        """.format(i[1], build_table(pivot_data,"red_dark",
                width="auto",
                font_family="sans-serif",
                font_size="13px",
                text_align="center",
                index=True
            ), month)

        part1 = MIMEText(html, 'html')
        msg.attach(part1)

    # list email recipient
        to = ['xxx@gmail.com']

        context = ssl.create_default_context()
        with smtplib.SMTP('smtp.gmail.com', 587) as server:
            server.ehlo()
            server.starttls(context=context)
            server.ehlo()
            server.login('xxx@gmail.com', 'xxx')
            server.sendmail('xxx@gmail.com',to, msg.as_string())
        print('>> Email SENT')
        print(' ')
        return 'kirim email'

### Send Email

In [ ]:
for i in list(list_shipper):
  data_shipper = raw_data[raw_data['global_shipper_id'] == i[0]]
  pivot_data = pd.pivot_table(data_shipper, index=['granular_status'], values=['tracking_id'], aggfunc='count')

# jika 0 order maka tidak ada pivotnya
  if data_shipper.shape[0] == 0:
    pass
  else:
    pivot_data['% of tracking_id'] = round((pivot_data['tracking_id']/pivot_data['tracking_id'].sum())*100,2)
    pivot_data['% of tracking_id'] = pivot_data['% of tracking_id'].map('{:.2f}%'.format)
    total_row = pd.DataFrame({
    'tracking_id': [pivot_data['tracking_id'].sum()],
    '% of tracking_id': ['100%']}, index=['Grand Total'])
    pivot_data = pd.concat([pivot_data,total_row], ignore_index=False)

  filenames = 'Monthly Report '+i[1]+' '+month+ '.xlsx'

# save as excel
  with pd.ExcelWriter(filenames) as writer:
    data_shipper.to_excel(writer, sheet_name='Raw Data',index=False)
    pivot_data.to_excel(writer, sheet_name='Summary')
  print(i[0], i[1], "-", "Total Order : ", data_shipper.shape[0])

# ##################################################################
# send email
 # kalo gaada ordernya, gausah kirim email
  if data_shipper.shape[0] == 0:
    print('>> 0 orders. Email NOT SENT')
    print(' ')
    pass
  else:
    send_email()